In [ ]:
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
import time

app = FastAPI()

def fake_stream_answer(query):
    answer = f"### Answer for: {query}\n\n- This is streamed\n- Token by token\n\n```python\nprint('Hello AI')\n```"

    for word in answer.split(" "):
        yield word + " "
        time.sleep(0.05)

@app.post("/ask")
async def ask(data: dict):
    query = data.get("query")

    return StreamingResponse(
        fake_stream_answer(query),
        media_type="text/plain"
    )

In [ ]:
npx create-next-app@latest ai-chat
cd ai-chat
npm install react-markdown

In [ ]:
"use client";

import { useState, useRef } from "react";
import ReactMarkdown from "react-markdown";

export default function Home() {
  const [messages, setMessages] = useState<any[]>([]);
  const [input, setInput] = useState("");
  const [loading, setLoading] = useState(false);
  const [error, setError] = useState("");

  const controllerRef = useRef<any>(null);

  const sendMessage = async () => {
    if (!input) return;

    setError("");
    setLoading(true);

    const userMessage = { role: "user", content: input };
    const aiMessage = { role: "assistant", content: "" };

    setMessages((prev) => [...prev, userMessage, aiMessage]);
    setInput("");

    try {
      const response = await fetch("http://localhost:8000/ask", {
        method: "POST",
        body: JSON.stringify({ query: input }),
        headers: {
          "Content-Type": "application/json",
        },
      });

      if (!response.ok) {
        throw new Error("Server error. Please try again.");
      }

      const reader = response.body?.getReader();
      const decoder = new TextDecoder();

      let done = false;

      while (!done) {
        const { value, done: doneReading } = await reader!.read();
        done = doneReading;

        const chunk = decoder.decode(value);

        setMessages((prev) => {
          const updated = [...prev];
          updated[updated.length - 1].content += chunk;
          return updated;
        });
      }
    } catch (err: any) {
      setError(err.message || "Something went wrong");
    } finally {
      setLoading(false);
    }
  };

  return (
    <div className="flex flex-col h-screen max-w-2xl mx-auto p-4">

      {/* Chat Area */}
      <div className="flex-1 overflow-y-auto space-y-4">
        {messages.map((msg, i) => (
          <div
            key={i}
            className={`p-3 rounded ${
              msg.role === "user" ? "bg-blue-100" : "bg-gray-100"
            }`}
          >
            <ReactMarkdown>{msg.content}</ReactMarkdown>
          </div>
        ))}

        {loading && <p className="text-gray-500">AI is typing...</p>}
      </div>

      {/* Error */}
      {error && (
        <div className="text-red-500 text-sm mb-2">
          ⚠️ {error}
        </div>
      )}

      {/* Input */}
      <div className="flex gap-2 mt-2">
        <input
          className="flex-1 border p-2 rounded"
          value={input}
          onChange={(e) => setInput(e.target.value)}
          placeholder="Ask something..."
        />
        <button
          onClick={sendMessage}
          className="bg-black text-white px-4 rounded"
        >
          Send
        </button>
      </div>
    </div>
  );
}

In [ ]:
import ReactMarkdown from "react-markdown";

In [ ]:
const aiMessage = {
  role: "assistant",
  content: "",
  sources: ["doc1.pdf", "wiki.com/page"]
};

In [ ]:
{msg.sources && (
  <details className="mt-2 text-sm">
    <summary className="cursor-pointer text-blue-500">
      Sources
    </summary>
    <ul className="list-disc ml-4">
      {msg.sources.map((s: string, i: number) => (
        <li key={i}>{s}</li>
      ))}
    </ul>
  </details>
)}

In [ ]:
{loading && <p>AI is typing...</p>}

In [ ]:
if (!response.ok) {
  throw new Error("Server error. Please try again.");
}

In [ ]:
className="flex flex-col h-screen max-w-2xl mx-auto p-4"

In [ ]:
<input className="flex-1 border p-2 rounded text-sm sm:text-base" />

In [ ]:
const [messages, setMessages] = useState([]);

In [ ]:
useEffect(() => {
  localStorage.setItem("chat", JSON.stringify(messages));
}, [messages]);